# Caso B — Combos · Modelamiento desplegado

> **Qué hace este notebook.** Despliega el mismo modelamiento de `run_case_b`: perfil de tiendas, selección de k y comparación de algoritmos de clustering, ajuste del K-Means, reglas de asociación FP-Growth, combos por cluster y grafo de co-compra.

> **Aprendizaje no supervisado:** no hay variable objetivo; se busca estructura (segmentos) y patrones (co-compra).

> **Cómo ejecutar.** `Restart & Run All`; requiere el extra `caso_b`.

## 1. Datos: líneas de ticket y matriz de cestas

In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from tostao_ml.cases import masters

PROJECT = Path.cwd().parents[1] if Path.cwd().name.startswith('caso') else Path.cwd()
bootstrap_project(PROJECT)
with KedroSession.create(project_path=PROJECT) as session:
    catalog = session.load_context().catalog
    master_b, _ = masters.build_master_b(
        catalog.load('b_tickets'), catalog.load('b_detalle_tickets'),
        catalog.load('b_catalogo_productos'))
baskets = masters.build_baskets_b(master_b)
print('Líneas:', master_b.shape, '| Cestas (ticket x producto):', baskets.shape)

## 2. Perfil de compra por tienda

Mezcla de categorías (share) + estadísticos de cesta (n_tickets, items/ticket, ticket medio). Es la representación que se agrupa.

In [ ]:
from tostao_ml.framework.io import set_global_seed
from tostao_ml.cases.caso_b import build_store_profiles

set_global_seed(42)
profiles = build_store_profiles(master_b)
profiles.round(3)

## 3. ¿Cuántos clusters? Selección de k y comparación de algoritmos

Se estandariza el perfil y se barre k comparando **K-Means vs. Aglomerativo** por silhouette; k y algoritmo salen de los datos, no a dedo.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

X = StandardScaler().fit_transform(profiles)
rows = []
for k in (2, 3, 4, 5, 6):
    if k >= len(profiles):
        continue
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
    ag = AgglomerativeClustering(n_clusters=k).fit_predict(X)
    rows.append({'algoritmo': 'K-Means', 'k': k, 'silhouette': round(float(silhouette_score(X, km)), 4)})
    rows.append({'algoritmo': 'Aglomerativo', 'k': k, 'silhouette': round(float(silhouette_score(X, ag)), 4)})
comparacion = pd.DataFrame(rows).sort_values('silhouette', ascending=False).reset_index(drop=True)
comparacion

## 4. El modelo: K-Means del framework con el mejor k

In [ ]:
from tostao_ml.framework.models import KMeansModel

best_k = int(comparacion.iloc[0]['k'])
Xdf = pd.DataFrame(X, index=profiles.index, columns=profiles.columns)
kmodel = KMeansModel(n_clusters=best_k, random_state=42).fit(Xdf)
clusters = pd.Series(kmodel.predict(Xdf), index=profiles.index, name='cluster')
silhouette = float(kmodel.metadata.extra.get('silhouette', 0.0))
print(f'k* = {best_k}  silhouette = {silhouette:.3f}')
profiles.join(clusters).round(3)

## 5. Reglas de asociación (FP-Growth)

Sobre la matriz de cestas booleana; se filtran por lift>1 y soporte mínimo, y se restringen a pares (un antecedente, un consecuente).

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

itemsets = fpgrowth(baskets, min_support=0.02, use_colnames=True)
reglas = association_rules(itemsets, metric='lift', min_threshold=1.0)
reglas = reglas[(reglas['antecedents'].map(len) == 1) & (reglas['consequents'].map(len) == 1)]
reglas = reglas.sort_values('lift', ascending=False).reset_index(drop=True)
reglas_fmt = reglas.assign(
    antecedente=reglas['antecedents'].map(lambda s: ', '.join(map(str, s))),
    consecuente=reglas['consequents'].map(lambda s: ', '.join(map(str, s))))
reglas_fmt[['antecedente', 'consecuente', 'support', 'confidence', 'lift']].head(20).round(4)

## 6. Combos por cluster (Top-N con precio y lift)

Por cada cluster se toman los pares de mayor lift y se propone un precio con descuento.

In [ ]:
from tostao_ml.cases.caso_b import top_combos_per_cluster
from tostao_ml.framework.viz import eda as eda_viz

combos = top_combos_per_cluster(master_b, baskets, clusters, top_n=5)
if not combos.empty:
    top = combos.sort_values('lift', ascending=False).head(12)
    combo_lift = top.assign(combo=top['producto_a'] + ' + ' + top['producto_b']).set_index('combo')['lift']
    eda_viz.pareto(combo_lift, name='combo (lift)').show()
combos.head(15)

## 7. Grafo de co-compra (enfoque complementario)

Productos «hub» por centralidad de eigenvector: conectan la red de co-compra.

In [ ]:
from tostao_ml.cases.caso_b import copurchase_graph_centrality

name_map = master_b.groupby('id_producto', observed=True)['nombre'].first()
copurchase_graph_centrality(baskets, name_map)

## Conclusión

La segmentación es moderada pero clara y las reglas tienen lift alto con soporte suficiente: los combos reflejan co-compra genuina y se priorizan por segmento. Mismo modelamiento que ejecuta el pipeline `caso_b`.